# Entanglement sources analisys using Genqo

The work in this notebook is based on the paper "Entanglement distribution: To herald or not to herald" by Jeffrey H. Shapiro, Clark Embleton, J. Gabriel Richardson.
we will show that Genqo can be used to efficienntly simulate 

ZALM using Genqo

In [95]:
using Genqo
using Gabs
using QuantumOpticsBase
using LinearAlgebra
import LinearAlgebra: tr, dot, norm
using Plots
import Plots: plot, surface


bell fraction

First calculate the Bell-state probability:

In [96]:
function zalm_Pbell_numerical(μ::Float64, ηR::Float64, ηT::Float64; engine::HybridProjectionEngine)
    st = eprstate(QuadBlockBasis(8), asinh(√μ), 0.)
    apply!(st, [2,4,5,7], modeswap(QuadBlockBasis(4)))
    apply!(st, [3,5,4,6], beamsplitter(QuadBlockBasis(4), 0.5))

    η = [ηR,ηR,ηT,ηT,ηT,ηT,ηR,ηR]
    Π = projector([:,:,1,1,0,0,:,:])
    st = project(st, Π; engine, η=η)

    ψ⁺ = (clicks([1,0,0,1]) + clicks([0,1,1,0])) / √2
    ψ⁻ = (clicks([1,0,0,1]) - clicks([0,1,1,0])) / √2
    ϕ⁺ = (clicks([1,0,1,0]) + clicks([0,1,0,1])) / √2
    ϕ⁻ = (clicks([1,0,1,0]) - clicks([0,1,0,1])) / √2

    real(dot(ψ⁺', st, ψ⁺) + dot(ψ⁻', st, ψ⁻) + dot(ϕ⁺', st, ϕ⁺) + dot(ϕ⁻', st, ϕ⁻))
end

zalm_Pbell_numerical (generic function with 1 method)

In [97]:
function zalm_Pload_numerical(μ::Float64, ηR::Float64, ηT::Float64; engine::HybridProjectionEngine)
    st = eprstate(QuadBlockBasis(8), asinh(√μ), 0.)
    apply!(st, [2,4, 5,7], modeswap(QuadBlockBasis(4)))
    apply!(st, [3,5, 4,6], beamsplitter(QuadBlockBasis(4), 0.5))

    η = [ηR,ηR,ηT,ηT,ηT,ηT,ηR,ηR]
    Π_S = projector([:,:,1,1,0,0,:,:])
    Π_A0 = projector([0,0,1,1,0,0,:,:])
    Π_B0 = projector([:,:,1,1,0,0,0,0])
    Π_S0 = projector([0,0,1,1,0,0,0,0])
    st_S = project(st, Π_S; engine, η=η)
    st_A0 = project(st, Π_A0; engine, η=η)
    st_B0 = project(st, Π_B0; engine, η=η)
    st_S0 = project(st, Π_S0; engine, η=η)

    tr(st_S) - tr(st_A0) - tr(st_B0) + tr(st_S0)
end


zalm_Pload_numerical (generic function with 1 method)

In [98]:
function zalm_bell_fraction_symbolic(μ::Real, ηR::Real, ηT::Real)::Real
        Ns  = (ηT*μ + 1) / (μ + 1)

        N′s = (ηR/Ns + (1 - ηR))^(-1)

        Pr_loadable = 1 - 2*(N′s^2)*(1 - (ηR*N′s)/(2*Ns))^2 + (N′s^4)*(1 - (ηR*N′s)/Ns)^2
        Pr_Bell = 2*N′s^4 * (
            2*(1 - N′s)^2
            - (2*ηR*(3*N′s^3 - 5*N′s^2 + 2*N′s)) / Ns
            + (ηR^2*(4*N′s^4 - 6*N′s^3 + 2*N′s^2)) / Ns^2
        ) + (ηR^2 * N′s^8) / (2*Ns^2)
        
        return Pr_Bell / Pr_loadable
    end

zalm_bell_fraction_symbolic (generic function with 1 method)

In [99]:
function plot_zalm2_Bell_state_fraction()
    engine = HybridProjectionEngine(8)
    μ = logrange(1e-4, 1.5, 100)

    Pload = zalm_Pload_numerical.(μ, 0.01, 0.9; engine=engine)
    PBell = zalm_Pbell_numerical.(μ, 0.01, 0.9; engine=engine)
    numerical_bell_fraction = PBell ./ Pload
    Plots.plot(μ, zalm_bell_fraction_symbolic.(μ, 0.01, 0.9), label="Symbolic", xlabel="Mean Photon Number Per Mode", ylabel="Bell-state Fraction", legend=:topright,color=1, title = "Bell-state Fraction vs\nMean Photon Number Per Mode")
    Plots.plot!(μ, numerical_bell_fraction, label="Numerical (Genqo v2.0)", color=2, linestyle=:dash)
end
@time plot_zalm2_Bell_state_fraction()


MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...


Fidelity

fidelity is p phi- over pbell

In [100]:
function zalm_bell_fidelity_numerical(μ::Float64,ηR::Float64, ηT::Float64; engine::HybridProjectionEngine)
        st = eprstate(QuadBlockBasis(8), asinh(√μ), 0.)
        apply!(st,  [2,4, 5,7], modeswap(QuadBlockBasis(4)))
        apply!(st, [3,5, 4,6], beamsplitter(QuadBlockBasis(4), 0.5))

        η = [ηR,ηR,ηT,ηT,ηT,ηT,ηR,ηR]
        Π = projector([:,:,1,1,0,0,:,:])
        st = project(st, Π; engine, η=η)

        ψ⁺ = (clicks([1,0,0,1]) + clicks([0,1,1,0])) / √2
        ψ⁻ = (clicks([1,0,0,1]) - clicks([0,1,1,0])) / √2
        ϕ⁺ = (clicks([1,0,1,0]) + clicks([0,1,0,1])) / √2
        ϕ⁻ = (clicks([1,0,1,0]) - clicks([0,1,0,1])) / √2

       real(dot(ψ⁺', st, ψ⁺)) / real(dot(ψ⁺', st, ψ⁺) + dot(ψ⁻', st, ψ⁻) + dot(ϕ⁺', st, ϕ⁺) + dot(ϕ⁻', st, ϕ⁻))
    end

zalm_bell_fidelity_numerical (generic function with 1 method)

In [101]:
 function zalm_bell_fidelity_symbolic(μ::Real, ηR::Real, ηT::Real)::Real

        Ns  = (ηT*μ + 1) / (μ + 1)
        
        N′s = (ηR/Ns + (1 - ηR))^(-1)
        
        Pr_psi_minus = (N′s^4 / 2) * (
            2*(1 - N′s)^2
            - (2*ηR*(3*N′s^3 - 5*N′s^2 + 2*N′s)) / Ns
            + (ηR^2*(5*N′s^4 - 6*N′s^3 + 2*N′s^2)) / Ns^2  
        )
        Pr_Bell = 2*N′s^4 * (
            2*(1 - N′s)^2
            - (2*ηR*(3*N′s^3 - 5*N′s^2 + 2*N′s)) / Ns
            + (ηR^2*(4*N′s^4 - 6*N′s^3 + 2*N′s^2)) / Ns^2
        ) + (ηR^2 * N′s^8) / (2*Ns^2)

        return Pr_psi_minus / Pr_Bell
    end

zalm_bell_fidelity_symbolic (generic function with 1 method)

In [102]:

function plot_zalm2_bell_fidelity()
    engine = HybridProjectionEngine(8)
    μ = logrange(1e-4, 1.5, 100)

    F_symbolic = zalm_bell_fidelity_symbolic.(μ, 0.01, 0.9)
    F_numerical = zalm_bell_fidelity_numerical.(μ, 0.01, 0.9; engine=engine)
    Plots.plot(μ, F_symbolic, label="Symbolic", xlabel="Mean Photon Number Per Mode", ylabel="Bell-state Fidelity", legend=:topright,color=1, title = "Bell-state Fidelity vs\nMean Photon Number Per Mode")
    Plots.plot!(μ, F_numerical, label="Numerical (Genqo v2.0)", color=2, linestyle=:dash)
end
@time plot_zalm2_bell_fidelity()


MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...


SPDC unheralded source

bell fraction

In [103]:
 function spdc_Pload_numerical(μ::Float64, ηR::Float64; engine::HybridProjectionEngine)
    st = eprstate(QuadBlockBasis(4), asinh(√μ), 0.)
    apply!(st,[2,4], modeswap(QuadBlockBasis(2)))
    η = [ηR,ηR,ηR,ηR]
    Π_A0 = projector([0,0,:,:])
    Π_B0 = projector([:,:,0,0])
    Π_S0 = projector([0,0,0,0])

    st_A0 = project(st, Π_A0; engine, η=η)
    st_B0 = project(st, Π_B0; engine, η=η)
    st_S0 = project(st, Π_S0; engine, η=η)

    1 - tr(st_A0) - tr(st_B0) + tr(st_S0)
end

spdc_Pload_numerical (generic function with 1 method)

In [104]:
function spdc_Pbell_numerical(μ::Float64, ηR::Float64; engine::HybridProjectionEngine)
    st = eprstate(QuadBlockBasis(4), asinh(√μ), 0.)
    apply!(st, [2,4], modeswap(QuadBlockBasis(2)))
    η = [ηR,ηR,ηR,ηR]
    Π = projector([:,:,:,:])
    st = project(st, Π; engine, η=η)

    ψ⁺ = (clicks([1,0,0,1]) + clicks([0,1,1,0])) / √2
    ψ⁻ = (clicks([1,0,0,1]) - clicks([0,1,1,0])) / √2
    ϕ⁺ = (clicks([1,0,1,0]) + clicks([0,1,0,1])) / √2
    ϕ⁻ = (clicks([1,0,1,0]) - clicks([0,1,0,1])) / √2

    real(dot(ψ⁺', st, ψ⁺) + dot(ψ⁻', st, ψ⁻) + dot(ϕ⁺', st, ϕ⁺) + dot(ϕ⁻', st, ϕ⁻))
end

spdc_Pbell_numerical (generic function with 1 method)

In [105]:
function spdc_bell_fraction_symbolic(μ::Float64, ηR::Float64)::Float64
        G = μ + 1 
        Pr_loadable = 1 - 2/(ηR*μ + 1)^2 + 1/(ηR*(2 - ηR)*μ + 1)^2
        Pr_Bell = (ηR^2 * μ * (3*G - 1 + ηR*(ηR - 2)*μ) + 3*(ηR*(ηR - 1)*μ)^2) / ((ηR*(ηR - 2)*μ - 1)^4)
        return Pr_Bell / Pr_loadable
    end


spdc_bell_fraction_symbolic (generic function with 1 method)

In [106]:

function plot_spdc1_bell_fraction()
    engine = HybridProjectionEngine(4)
    μ = logrange(1e-4, 1.5, 100)

    Pbell = spdc_Pbell_numerical.(μ, 0.01; engine=engine)
    Pload = spdc_Pload_numerical.(μ, 0.01; engine=engine)
    bell_fraction_numerical =  Pbell ./ Pload
    bell_fraction_symbolic = spdc_bell_fraction_symbolic.(μ, 0.01)

    Plots.plot(μ, bell_fraction_symbolic, label="symbolic", xlabel="Mean Photon Number Per Mode", ylabel="Bell-state Fraction", legend=:topright,color=1)
    Plots.plot!(μ, bell_fraction_numerical, label="numerical(Genqo v2.0)", linestyle=:dash, color=[:blue])
end
@time plot_spdc1_bell_fraction()

MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...


fidelity

In [107]:
function spdc_bell_fidelity_numerical(μ::Float64, ηR::Float64; engine::HybridProjectionEngine)
    st = eprstate(QuadBlockBasis(4), asinh(√μ), 0.)
    apply!(st, [2,4], modeswap(QuadBlockBasis(2)))
    η = [ηR,ηR,ηR,ηR]
    Π = projector([:,:,:,:])
    st = project(st, Π; engine, η=η)
    

    ψ⁺ = (clicks([1,0,0,1]) + clicks([0,1,1,0])) / √2
    ψ⁻ = (clicks([1,0,0,1]) - clicks([0,1,1,0])) / √2
    ϕ⁺ = (clicks([1,0,1,0]) + clicks([0,1,0,1])) / √2
    ϕ⁻ = (clicks([1,0,1,0]) - clicks([0,1,0,1])) / √2

    real(dot(ψ⁺', st, ψ⁺))/ real(dot(ψ⁺', st, ψ⁺) + dot(ψ⁻', st, ψ⁻) + dot(ϕ⁺', st, ϕ⁺) + dot(ϕ⁻', st, ϕ⁻))
end

spdc_bell_fidelity_numerical (generic function with 1 method)

In [108]:
function spdc_bell_fidelity_symbolic(μ::Float64, ηR::Float64)::Float64
        G = μ + 1 
        Pr_psi_minus = (ηR^2 * μ * (3*G - 1 + ηR*(ηR - 2)*μ)) / ((ηR*(ηR - 2)*μ - 1)^4)
        Pr_Bell = (ηR^2 * μ * (3*G - 1 + ηR*(ηR - 2)*μ) + 3*(ηR*(ηR - 1)*μ)^2) / ((ηR*(ηR - 2)*μ - 1)^4)
        return Pr_psi_minus / Pr_Bell
    end

spdc_bell_fidelity_symbolic (generic function with 1 method)

In [109]:
function plot_spdc1_bell_fidelity()
    engine = HybridProjectionEngine(4)
    μ = logrange(1e-4, 1.5, 100)

    F_symbolic = spdc_bell_fidelity_symbolic.(μ, 0.01)
    F_numerical = spdc_bell_fidelity_numerical.(μ, 0.01; engine=engine)
    Plots.plot(μ, F_symbolic, label="Symbolic", xlabel="Mean Photon Number Per Mode", ylabel="Fidelity", legend=:topright,color=1)
    Plots.plot!(μ, F_numerical, label="Numerical (Genqo v2.0)", color=2, linestyle=:dash)
end
@time plot_spdc1_bell_fidelity()

MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...


Chahine et al source

In [110]:
function chahine_Pbell_numerical(μ::Float64, ηR::Float64, ηT::Float64; engine::HybridProjectionEngine)
    st = eprstate(QuadBlockBasis(4), asinh(√μ), 0.) ⊗ vacuumstate(QuadBlockBasis(2))
    apply!(st, [2,4], modeswap(QuadBlockBasis(2)))
    apply!(st, [3,5, 4,6], beamsplitter(QuadBlockBasis(4), 0.5))

    η = [ηT,ηT,ηR,ηR,ηR,ηR]
    Π_S = projector([1,1,:,:,:,:])

    st = project(st, Π_S; engine=engine, η=η)
    ψ⁺ = (clicks([1,0,0,1]) + clicks([0,1,1,0])) / √2
    ψ⁻ = (clicks([1,0,0,1]) - clicks([0,1,1,0])) / √2
    ϕ⁺ = (clicks([1,0,1,0]) + clicks([0,1,0,1])) / √2
    ϕ⁻ = (clicks([1,0,1,0]) - clicks([0,1,0,1])) / √2

    real(dot(ψ⁺', st, ψ⁺) + dot(ψ⁻', st, ψ⁻) + dot(ϕ⁺', st, ϕ⁺) + dot(ϕ⁻', st, ϕ⁻))
end

chahine_Pbell_numerical (generic function with 1 method)

In [111]:
 function chahine_Pload_numerical(μ::Float64, ηR::Float64, ηT::Float64; engine::HybridProjectionEngine)
        st = eprstate(QuadBlockBasis(4), asinh(√μ), 0.) ⊗ vacuumstate(QuadBlockBasis(2))
        apply!(st,[2,4], modeswap(QuadBlockBasis(2)))
        apply!(st, [3,5, 4,6], beamsplitter(QuadBlockBasis(4), 0.5))

        η = [ηT,ηT,ηR,ηR,ηR,ηR]
        Π_S = projector([1,1,:,:,:,:])
        Π_A0 = projector([1,1,0,0,:,:])
        Π_B0 = projector([1,1,:,:,0,0])
        Π_S0 = projector([1,1,0,0,0,0])

        st_S = project(st, Π_S; engine=engine, η=η)
        st_A0 = project(st, Π_A0; engine=engine, η=η)
        st_B0 = project(st, Π_B0; engine=engine, η=η)
        st_S0 = project(st, Π_S0; engine=engine, η=η)

        tr(st_S) - tr(st_A0) - tr(st_B0) + tr(st_S0)
    end

chahine_Pload_numerical (generic function with 1 method)

In [112]:
    function chahine_bell_faction_symbolic(μ::Float64, ηR::Float64, ηT::Float64)::Float64
        Ns  = (ηT*μ + 1) / (μ + 1)
        N′s = (ηR/Ns + (1 - ηR))^(-1)
        Ñs = 2*N′s/(N′s + 1)
        Pr_loadable = 1 - 2*(Ñs^2)*(1 - (ηR*Ñs)/(2*Ns))^2 + (N′s^2)*(1 - (ηR*N′s)/Ns)^2
        
        Pr_Bell = N′s^2 * (
            3*(1 - N′s)^2 / 2
            - (3*ηR*N′s*(1 - 2*N′s)*(1 - N′s)) / Ns
            + (ηR^2*N′s^2*((1 - 2*N′s)^2 + 2*(1 - 3*N′s)*(1 - N′s))) / (2*Ns^2)
        )
        return Pr_Bell / Pr_loadable
    end

chahine_bell_faction_symbolic (generic function with 1 method)

In [113]:
function plot_chahine_bell_fraction()
    engine = HybridProjectionEngine(6)
    μ = logrange(1e-4, 1.5, 100)

    bell_faction_symbolic = chahine_bell_faction_symbolic.(μ, 0.01, 0.9)
    
    bell_probability_numerical = chahine_Pbell_numerical.(μ, 0.01, 0.9; engine=engine)
    loadable_probability_numerical = chahine_Pload_numerical.(μ, 0.01, 0.9; engine=engine)
    bell_faction_numerical = bell_probability_numerical ./ loadable_probability_numerical
    
    Plots.plot(μ, bell_faction_symbolic, label="Symbolic", xlabel="Mean Photon Number Per Mode", ylabel="Bell-state Fraction", legend=:topright,color=1)
    Plots.plot!(μ, bell_faction_numerical, label="Numerical (Genqo v2.0)", color=2, linestyle=:dash)
end
@time plot_chahine_bell_fraction()

MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...


Fidelity

In [114]:
  function chahine_bell_fidelity_numerical(μ::Float64, ηR::Float64, ηT::Float64; engine::HybridProjectionEngine)
        st = eprstate(QuadBlockBasis(4), asinh(√μ), 0.) ⊗ vacuumstate(QuadBlockBasis(2))
        apply!(st,[2,4], modeswap(QuadBlockBasis(2)))
        apply!(st, [3,5,4,6], beamsplitter(QuadBlockBasis(4), 0.5))

        η = [ηT,ηT,ηR,ηR,ηR,ηR]
        Π = projector([1,1,:,:,:,:])
        st = project(st, Π; engine=engine, η=η)

        ψ⁺ = (clicks([1,0,0,1]) + clicks([0,1,1,0])) / √2
        ψ⁻ = (clicks([1,0,0,1]) - clicks([0,1,1,0])) / √2
        ϕ⁺ = (clicks([1,0,1,0]) + clicks([0,1,0,1])) / √2
        ϕ⁻ = (clicks([1,0,1,0]) - clicks([0,1,0,1])) / √2

       real(dot(ψ⁺', st, ψ⁺)) / real(dot(ψ⁺', st, ψ⁺) + dot(ψ⁻', st, ψ⁻) + dot(ϕ⁺', st, ϕ⁺) + dot(ϕ⁻', st, ϕ⁻))
    end

chahine_bell_fidelity_numerical (generic function with 1 method)

In [115]:
function chahine_bell_fidelity_symbolic(μ::Real , ηR::Real , ηT::Real)::Real
        Ns  = (ηT*μ + 1) / (μ + 1)
        N′s = (ηR/Ns + (1 - ηR))^(-1)
        
        Pr_psi_minus = N′s^2 * (
            (1 - N′s)^2 / 2
            - (ηR*N′s*(1 - 2*N′s)*(1 - N′s)) / Ns
            + (ηR^2*N′s^2*(1 - 2*N′s)^2) / (2*Ns^2)
        )
    
        Pr_Bell = N′s^2 * (
            3*(1 - N′s)^2 / 2
            - (3*ηR*N′s*(1 - 2*N′s)*(1 - N′s)) / Ns
            + (ηR^2*N′s^2*((1 - 2*N′s)^2 + 2*(1 - 3*N′s)*(1 - N′s))) / (2*Ns^2)
        )
    
        return Pr_psi_minus / Pr_Bell
    end


chahine_bell_fidelity_symbolic (generic function with 1 method)

In [116]:
function plot_chahine_bell_fidelity()
    engine = HybridProjectionEngine(6)
    μ = logrange(1e-4, 1.5, 100)

    F_symbolic = chahine_bell_fidelity_symbolic.(μ, 0.01, 0.9)
    F_numerical = chahine_bell_fidelity_numerical.(μ, 0.01, 0.9; engine=engine)
    Plots.plot(μ, F_symbolic, label="symbolic", xlabel="Mean Photon Number Per Mode", ylabel="Fidelity", legend=:topright,color=1)
    Plots.plot!(μ, F_numerical, label="numerical - (Genqo v2.0)", color=2, linestyle=:dash)
end
@time plot_chahine_bell_fidelity()

MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...


Compare all sources.

In [117]:
function plot_bell_state_fraction_comparison()
    engine_spdc = HybridProjectionEngine(4)
    engine_zalm = HybridProjectionEngine(8)
    engine_chahine = HybridProjectionEngine(6)
    μ = logrange(1e-4, 1.5, 100)

    ηR = 0.01
    ηT = 0.9

    Bfraction_zalm_numerical = zalm_Pbell_numerical.(μ, ηR, ηT; engine=engine_zalm) ./ zalm_Pload_numerical.(μ, ηR, ηT; engine=engine_zalm)
    Bfraction_zalm_symbolic = zalm_bell_fraction_symbolic.(μ, ηR, ηT)

    Bfraction_chahine_numerical = chahine_Pbell_numerical.(μ, ηR, ηT; engine=engine_chahine) ./ chahine_Pload_numerical.(μ, ηR, ηT; engine=engine_chahine)
    Bfraction_chahine_symbolic = chahine_bell_faction_symbolic.(μ, ηR, ηT)

    Bfraction_spdc_symbolic = spdc_bell_fraction_symbolic.(μ, ηR)
    Bfraction_spdc_numerical = spdc_Pbell_numerical.(μ, 0.01; engine=engine_spdc) ./ spdc_Pload_numerical.(μ, 0.01; engine=engine_spdc)


    Plots.plot(μ, Bfraction_zalm_symbolic, label = "ZALM analytic", xlabel = "Mean Photon Number",ylabel = "Bell-state Fraction", legend = :bottomleft,color = 1)
    Plots.plot!(μ, Bfraction_chahine_symbolic, label = "Chahine analytic", color = 3)
    Plots.plot!(μ, Bfraction_spdc_symbolic, label = "SPDC analytic", color = 4)
    Plots.plot!(μ, Bfraction_zalm_numerical, label = "ZALM numerical (Genqo v2.0)", color = 2, linestyle = :dash)
    Plots.plot!(μ, Bfraction_chahine_numerical, label = "Chahine numerical (Genqo v2.0)", color = 5, linestyle = :dash)
    Plots.plot!(μ, Bfraction_spdc_numerical, label = "SPDC numerical (Genqo v2.0)", color = 6, linestyle = :dash)
    Plots.title!("Bell-state Fraction Comparison for: \$\\eta_R = $(ηR),\\; \\eta_T = $(ηT)\$")
end
@time plot_bell_state_fraction_comparison()

MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...


In [118]:
function plot_bell_state_fidelity_comparison()
    engine_spdc = HybridProjectionEngine(4)
    engine_zalm = HybridProjectionEngine(8)
    engine_chahine = HybridProjectionEngine(6)
    μ = logrange(1e-4, 1.5, 100)

    ηR = 0.01
    ηT = 0.9

    fidelity_zalm_numerical = zalm_bell_fidelity_numerical.(μ, ηR, ηT; engine=engine_zalm)
    fidelity_zalm_symbolic = zalm_bell_fidelity_symbolic.(μ, ηR, ηT)

    fidelity_chahine_numerical = chahine_bell_fidelity_numerical.(μ, ηR, ηT; engine=engine_chahine)
    fidelity_chahine_symbolic = chahine_bell_fidelity_symbolic.(μ, ηR, ηT)

    fidelity_spdc_symbolic = spdc_bell_fidelity_symbolic.(μ, ηR)
    fidelity_spdc_numerical = spdc_bell_fidelity_numerical.(μ, 0.01; engine=engine_spdc)


    Plots.plot(μ, fidelity_zalm_symbolic, label = "ZALM analytic", xlabel = "Mean Photon Number",ylabel = "Bell-state Fidelity", legend = :bottomleft,color = 1)
    Plots.plot!(μ, fidelity_chahine_symbolic, label = "Chahine analytic", color = 3)
    Plots.plot!(μ, fidelity_spdc_symbolic, label = "SPDC analytic", color = 4)
    Plots.plot!(μ, fidelity_zalm_numerical, label = "ZALM numerical (Genqo v2.0)", color = 2, linestyle = :dash)
    Plots.plot!(μ, fidelity_chahine_numerical, label = "Chahine numerical (Genqo v2.0)", color = 5, linestyle = :dash)
    Plots.plot!(μ, fidelity_spdc_numerical, label = "SPDC numerical (Genqo v2.0)", color = 6, linestyle = :dash)
    Plots.title!("Bell-state Fidelity Comparison for: \$\\eta_R = $(ηR),\\; \\eta_T = $(ηT)\$")
end
@time plot_bell_state_fidelity_comparison()

MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...


Now probability Hnm for zalm and chahine source

In [119]:
function zalm_PHeralded_symbolic(μ::Float64, ηR::Float64, ηT::Float64)::Float64
    PH = (4 * (ηT * μ)^2)/ ((ηT * μ) + 1)^6    
    return PH
end

zalm_PHeralded_symbolic (generic function with 1 method)

In [120]:
function zalm_PHeralded_numerical(μ::Float64, ηR::Float64, ηT::Float64; engine::HybridProjectionEngine)
    st = eprstate(QuadBlockBasis(8), asinh(√μ), 0.)
    apply!(st,[2,4, 5,7], modeswap(QuadBlockBasis(4)))
    apply!(st, [3,5, 4,6], beamsplitter(QuadBlockBasis(4), 0.5))

    η = [ηR,ηR,ηT,ηT,ηT,ηT,ηR,ηR]
    Π = projector([:,:,0,1,1,0,:,:])
    st = project(st, Π; engine=engine, η=η)

    return 4*tr(st)
end

zalm_PHeralded_numerical (generic function with 1 method)

In [121]:
function plot_zalm_PHeralded()
    engine = HybridProjectionEngine(8)
    μ = logrange(1e-4, 1.5, 100)

    PH_symbolic = zalm_PHeralded_symbolic.(μ, 0.01, 0.9)
    PH_numerical = zalm_PHeralded_numerical.(μ, 0.01, 0.9; engine=engine)
    Plots.plot(μ, PH_symbolic, label="Symbolic", xlabel="Mean Photon Number Per Mode", ylabel="Heralding Probability", legend=:topright,color=1)
    Plots.plot!(μ, PH_numerical, label="Numerical (Genqo v2.0)", color=2, linestyle=:dash)
end
@time plot_zalm_PHeralded()

MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...


Hnm probability chahine

In [122]:
function chahine_PHeralded_symbolic(μ::Float64, ηR::Float64, ηT::Float64)::Float64
    PH = ((ηT * μ)^2)/ ((ηT * μ) + 1)^4    
    return PH
end

chahine_PHeralded_symbolic (generic function with 1 method)

In [123]:
function chahine_PHeralded_numerical(μ::Float64, ηR::Float64, ηT::Float64; engine::HybridProjectionEngine)
    st = eprstate(QuadBlockBasis(4), asinh(√μ), 0.) ⊗ vacuumstate(QuadBlockBasis(2))
    apply!(st,[2,4], modeswap(QuadBlockBasis(2)))
    apply!(st, [3,5,4,6], beamsplitter(QuadBlockBasis(4), 0.5))

    η = [ηT,ηT,ηR,ηR,ηR,ηR]
    Π = projector([1,1,:,:,:,:])
    st = project(st, Π; engine=engine, η=η)

    return tr(st)
end

chahine_PHeralded_numerical (generic function with 1 method)

In [124]:
function plot_chahine_PHeralded()
    engine = HybridProjectionEngine(6)
    μ = logrange(1e-4, 1.5, 100)

    PH_symbolic = chahine_PHeralded_symbolic.(μ, 0.01, 0.9)
    PH_numerical = chahine_PHeralded_numerical.(μ, 0.01, 0.9; engine=engine)
    Plots.plot(μ, PH_symbolic, label="Symbolic", xlabel="Mean Photon Number Per Mode", ylabel="Heralding Probability", legend=:bottomright,color=1)
    Plots.plot!(μ, PH_numerical, label="Numerical (Genqo v2.0)", color=2, linestyle=:dash)
end
@time plot_chahine_PHeralded()

MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...


compare

In [125]:
function plot_Pherald_comparison()
    engine_zalm = HybridProjectionEngine(8)
    engine_chahine = HybridProjectionEngine(6)
    μ = logrange(1e-4, 1.5, 100)

    ηR = 0.01
    ηT = 0.9

    PH_zalm_numerical = zalm_PHeralded_numerical.(μ, ηR, ηT; engine=engine_zalm)
    PH_zalm_symbolic = zalm_PHeralded_symbolic.(μ, ηR, ηT)

    PH_chahine_numerical = chahine_PHeralded_numerical.(μ, ηR, ηT; engine=engine_chahine)
    PH_chahine_symbolic = chahine_PHeralded_symbolic.(μ, ηR, ηT)

    Plots.plot(μ, PH_zalm_symbolic, label = "ZALM analytic", xlabel = "Mean Photon Number",ylabel = "Heralding Probability", legend = :bottomright,color = 1)
    Plots.plot!(μ, PH_chahine_symbolic, label = "Chahine analytic", color = 3)
    Plots.plot!(μ, PH_zalm_numerical, label = "ZALM numerical (Genqo v2.0)", color = 2, linestyle = :dash)
    Plots.plot!(μ, PH_chahine_numerical, label = "Chahine numerical (Genqo v2.0)", color = 5, linestyle = :dash)
    Plots.title!("Heralding Probability Comparison for: \$\\eta_R = $(ηR),\\; \\eta_T = $(ηT)\$")
end
@time plot_Pherald_comparison()


MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...


Now calculate rate of distributed entanglement

In [126]:
function zalm_q_symbolic(μ::Float64, ηT::Float64)::Float64
    q = (2 * ηT * μ) / (ηT * μ + 1)^3
    return q
end

zalm_q_symbolic (generic function with 1 method)

In [127]:
function chahine_q_symbolic(μ::Float64, ηT::Float64)::Float64
    q = (ηT * μ) / (ηT * μ + 1)^2
    return q
end

chahine_q_symbolic (generic function with 1 method)

In [128]:
using SpecialFunctions: binomial

function pmf_zP(N_I::Int, q::Float64)
    # binomial pmf for z_H (or z_V)
    return [binomial(N_I, ℓ) * q^ℓ * (1 - q)^(N_I - ℓ) for ℓ in 0:N_I]
end

function p_k_array(N_I::Int, q::Float64)
    Pz = pmf_zP(N_I, q)
    p = zeros(Float64, N_I + 1)
    for k in 0:N_I
        term = Pz[k + 1]
        tail = sum(Pz[(k + 2):end])
        p[k + 1] = term * (term + 2 * tail)
    end
    return p
end

function E_H(p, N_M::Integer)
    N_I = length(p) - 1
    k = collect(0:N_I)
    N_M_eff = clamp(Int(N_M), 0, N_I)
    if N_M_eff <= 0
        return 0.0
    end
    return sum(k[1:N_M_eff] .* p[1:N_M_eff]) + N_M_eff * sum(p[(N_M_eff + 1):end])
end


E_H (generic function with 1 method)

In [129]:
function zalm_rate_dist_entanglement_numerical(μ::Float64, ηR::Float64, ηT::Float64, Rp::Float64, N_I::Integer, N_M; engine::HybridProjectionEngine)
    q = sqrt(zalm_PHeralded_numerical(μ, ηR, ηT; engine=engine))
    p_array = p_k_array(Int(N_I), q)
    EH = E_H(p_array, N_M)
    PBell = (zalm_Pbell_numerical(μ, ηR, ηT; engine=engine) ./ zalm_PHeralded_numerical(μ, ηR, ηT; engine=engine)) .* 4
    F = zalm_bell_fidelity_numerical(μ, ηR, ηT; engine=engine)

    return Rp * EH * PBell * F
end
function zalm_Pbell_symbolic(μ::Float64, ηR::Float64, ηT::Float64)
    Ns  = (ηT*μ + 1) / (μ + 1)
    N′s = (ηR/Ns + (1 - ηR))^(-1)
    Pr_Bell = 2*N′s^4 * (
        2*(1 - N′s)^2
        - (2*ηR*(3*N′s^3 - 5*N′s^2 + 2*N′s)) / Ns
        + (ηR^2*(4*N′s^4 - 6*N′s^3 + 2*N′s^2)) / Ns^2
    ) + (ηR^2 * N′s^8) / (2*Ns^2)
    
    return Pr_Bell
end
function zalm_rate_dist_entanglement_symbolic(μ::Float64, ηR::Float64, ηT::Float64, Rp, N_I::Integer, N_M::Integer)
    Pbell = zalm_Pbell_symbolic(μ, ηR, ηT)
    F = zalm_bell_fidelity_symbolic(μ, ηR, ηT)
    q = zalm_q_symbolic(μ, ηT)
    p_array = p_k_array(N_I, q)
    EH = E_H(p_array, N_M)

    return Rp * EH * F * Pbell
end  

zalm_rate_dist_entanglement_symbolic (generic function with 1 method)

In [130]:
function chahine_rate_dist_entanglement_numerical(μ::Float64, ηR::Float64, ηT::Float64, Rp::Float64, N_I::Integer, N_M; engine::HybridProjectionEngine)
    q = sqrt(chahine_PHeralded_numerical(μ, ηR, ηT; engine=engine))
    p_array = p_k_array(Int(N_I), q)
    EH = E_H(p_array, N_M)
    PBell = (chahine_Pbell_numerical(μ, ηR, ηT; engine=engine) ./ chahine_PHeralded_numerical(μ, ηR, ηT; engine=engine))
    F = chahine_bell_fidelity_numerical(μ, ηR, ηT; engine=engine)

    return Rp * EH * PBell * F
end

function chahine_Pbell_symbolic(μ::Float64, ηR::Float64, ηT::Float64)
    Ns  = (ηT*μ + 1) / (μ + 1)
    N′s = (ηR/Ns + (1 - ηR))^(-1)
    Ñs = 2*N′s/(N′s + 1)
    
    Pr_Bell = N′s^2 * (
        3*(1 - N′s)^2 / 2
        - (3*ηR*N′s*(1 - 2*N′s)*(1 - N′s)) / Ns
        + (ηR^2*N′s^2*((1 - 2*N′s)^2 + 2*(1 - 3*N′s)*(1 - N′s))) / (2*Ns^2)
    )
    return Pr_Bell
end
function chahine_rate_dist_entanglement_symbolic(μ::Float64, ηR::Float64, ηT::Float64, Rp, N_I::Integer, N_M::Integer)
    Pbell = chahine_Pbell_symbolic(μ, ηR, ηT)
    F = chahine_bell_fidelity_symbolic(μ, ηR, ηT)
    q = chahine_q_symbolic(μ, ηT)
    p_array = p_k_array(N_I, q)
    EH = E_H(p_array, N_M)

    return Rp * EH * F * Pbell
end 


chahine_rate_dist_entanglement_symbolic (generic function with 1 method)

In [131]:
function spdc_rate_dist_entanglement_numerical(μ::Float64, ηR::Float64, Rp::Float64, N_M; engine::HybridProjectionEngine)
    PBell = spdc_Pbell_numerical(μ, ηR; engine=engine)
    F = spdc_bell_fidelity_numerical(μ, ηR; engine=engine)
    return Rp * N_M * PBell * F
end
function spdc_Pbell_symbolic(μ::Float64, ηR::Float64)::Float64
    G = μ + 1 
    Pr_Bell = (ηR^2 * μ * (3*G - 1 + ηR*(ηR - 2)*μ) + 3*(ηR*(ηR - 1)*μ)^2) / ((ηR*(ηR - 2)*μ - 1)^4)
    return Pr_Bell
end
function spdc_rate_dist_entanglement_symbolic(μ::Float64, ηR::Float64, Rp::Float64, N_M::Int)
    PBell = spdc_Pbell_symbolic(μ, ηR)
    F = spdc_bell_fidelity_symbolic(μ, ηR)
    return Rp * N_M * PBell * F
end

spdc_rate_dist_entanglement_symbolic (generic function with 1 method)

In [132]:
function plot_rate_dist_entanglement()
    zalm_engine = HybridProjectionEngine(8)
    chahine_engine = HybridProjectionEngine(6)
    spdc_engine = HybridProjectionEngine(4)
    μ_zalm = 0.0173
    μ_chahine = 0.0263
    μ_spdc = 0.00694
    ηR = 0.01
    ηT = 0.9
    Rp = 10e9
    N_I = collect(1:50)
    N_M = 1

    zalm_rates_numerical = zalm_rate_dist_entanglement_numerical.(μ_zalm, ηR, ηT, Rp, N_I, N_M; engine=zalm_engine)
    zalm_rates_symbolic = zalm_rate_dist_entanglement_symbolic.(μ_zalm, ηR, ηT, Rp, N_I, N_M)

    chahine_rates_numerical = chahine_rate_dist_entanglement_numerical.(μ_chahine, ηR, ηT, Rp, N_I, N_M; engine=chahine_engine)
    chahine_rates_symbolic = chahine_rate_dist_entanglement_symbolic.(μ_chahine, ηR, ηT, Rp, N_I, N_M)
    
    spdc_rates_numerical = [spdc_rate_dist_entanglement_numerical(μ_spdc, ηR, Rp, N_M; engine=spdc_engine) for _ in N_I]
    spdc_rates_symbolic = [spdc_rate_dist_entanglement_symbolic(μ_spdc, ηR, Rp, N_M) for _ in N_I]

    Plots.plot(N_I, zalm_rates_numerical, seriestype=:scatter, label="ZALM Re (numerical)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", legend=:topleft,color=1, markersize=2)
    Plots.plot!(N_I, zalm_rates_symbolic, seriestype=:scatter, label="ZALM Re (symbolic)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)",color=2, markersize=2)
    Plots.plot!(N_I, chahine_rates_numerical, seriestype=:scatter, label="CHAHINE Re (numerical)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", color=3, markersize=2)
    Plots.plot!(N_I, chahine_rates_symbolic, seriestype=:scatter, label="CHAHINE Re (symbolic)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", color=4, markersize=2)
    Plots.plot!(N_I, spdc_rates_numerical, seriestype=:scatter, label="SPDC Re (numerical)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", color=5, markersize=2)
    Plots.plot!(N_I, spdc_rates_symbolic, seriestype=:scatter, label="SPDC Re (symbolic)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", color=6, markersize=2)
end
@time plot_rate_dist_entanglement()


MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...


Now add memories.

In [133]:
function plot_rate_dist_entanglement_with_Nm()
    zalm_engine = HybridProjectionEngine(8)
    chahine_engine = HybridProjectionEngine(6)
    spdc_engine = HybridProjectionEngine(4)
    μ_zalm = 0.0173
    μ_chahine = 0.0263
    μ_spdc = 0.00694
    ηR = 0.01
    ηT = 0.9
    Rp = 10e9
    N_I = collect(1:50)
    N_M = N_I
    zalm_rates_numerical = zalm_rate_dist_entanglement_numerical.(μ_zalm, ηR, ηT, Rp, N_I, N_M; engine=zalm_engine) 
    zalm_rates_symbolic = zalm_rate_dist_entanglement_symbolic.(μ_zalm, ηR, ηT, Rp, N_I, N_M)

    chahine_rates_numerical = chahine_rate_dist_entanglement_numerical.(μ_chahine, ηR, ηT, Rp, N_I, N_M; engine=chahine_engine)
    chahine_rates_symbolic = chahine_rate_dist_entanglement_symbolic.(μ_chahine, ηR, ηT, Rp, N_I, N_M)
    
    spdc_rates_numerical = [spdc_rate_dist_entanglement_numerical(μ_spdc, ηR, Rp, N_M_i; engine=spdc_engine) for N_M_i in N_M]
    spdc_rates_symbolic = [spdc_rate_dist_entanglement_symbolic(μ_spdc, ηR, Rp, N_M_i) for N_M_i in N_M]

    Plots.plot(N_I, zalm_rates_numerical, seriestype=:scatter, label="ZALM Re (numerical)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", legend=:topleft,color=1, markersize=3)
    Plots.plot!(N_I, zalm_rates_symbolic, seriestype=:scatter, label="ZALM Re (symbolic)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)",color=2, markersize=3)
    Plots.plot!(N_I, chahine_rates_numerical, seriestype=:scatter, label="CHAHINE Re (numerical)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", color=3, markersize=3)
    Plots.plot!(N_I, chahine_rates_symbolic, seriestype=:scatter, label="CHAHINE Re (symbolic)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", color=4, markersize=3)
    Plots.plot!(N_I, spdc_rates_numerical, seriestype=:scatter, label="SPDC Re (numerical)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", color=5, markersize=3)
    Plots.plot!(N_I, spdc_rates_symbolic, seriestype=:scatter, label="SPDC Re (symbolic)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", color=6, markersize=3)
end
@time plot_rate_dist_entanglement_with_Nm()


MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...


last plot before dark counts

In [134]:
function plot_rate_dist_entanglement_with_Nm()
    zalm_engine = HybridProjectionEngine(8)
    chahine_engine = HybridProjectionEngine(6)
    spdc_engine = HybridProjectionEngine(4)
    μ_zalm = 0.0173
    μ_chahine = 0.0263
    μ_spdc = 0.00694
    ηR = 0.01
    ηT = 0.75
    Rp = 10e9
    N_I = collect(5:50)
    N_M = 5

    zalm_rates_numerical = zalm_rate_dist_entanglement_numerical.(μ_zalm, ηR, ηT, Rp, N_I, N_M; engine=zalm_engine)
    zalm_rates_symbolic = zalm_rate_dist_entanglement_symbolic.(μ_zalm, ηR, ηT, Rp, N_I, N_M)

    chahine_rates_numerical = chahine_rate_dist_entanglement_numerical.(μ_chahine, ηR, ηT, Rp, N_I, N_M; engine=chahine_engine)
    chahine_rates_symbolic = chahine_rate_dist_entanglement_symbolic.(μ_chahine, ηR, ηT, Rp, N_I, N_M)
    
    spdc_rates_numerical = [spdc_rate_dist_entanglement_numerical(μ_spdc, ηR, Rp, N_M; engine=spdc_engine) for _ in N_I]
    spdc_rates_symbolic = [spdc_rate_dist_entanglement_symbolic(μ_spdc, ηR, Rp, N_M) for _ in N_I]

    Plots.plot(N_I, zalm_rates_numerical, seriestype=:scatter, label="ZALM Re (numerical)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", legend=:topleft,color=1, markersize=3)
    Plots.plot!(N_I, zalm_rates_symbolic, seriestype=:scatter, label="ZALM Re (symbolic)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)",color=2, markersize=3)
    Plots.plot!(N_I, chahine_rates_numerical, seriestype=:scatter, label="CHAHINE Re (numerical)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", color=3, markersize=3)
    Plots.plot!(N_I, chahine_rates_symbolic, seriestype=:scatter, label="CHAHINE Re (symbolic)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", color=4, markersize=3)
    Plots.plot!(N_I, spdc_rates_numerical, seriestype=:scatter, label="SPDC Re (numerical)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", color=5, markersize=3)
    Plots.plot!(N_I, spdc_rates_symbolic, seriestype=:scatter, label="SPDC Re (symbolic)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", color=6, markersize=3)
end
@time plot_rate_dist_entanglement_with_Nm()


MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...


In [135]:
function plot_rate_dist_entanglement_with_Nm()
    zalm_engine = HybridProjectionEngine(8)
    chahine_engine = HybridProjectionEngine(6)
    spdc_engine = HybridProjectionEngine(4)
    μ = logrange(1e-4, 1.5, 100)
    ηR = 0.01
    ηT = 0.9
    Rp = 10e9
    N_I = 10
    N_M = 1

    zalm_rates_numerical = zalm_rate_dist_entanglement_numerical.(μ, ηR, ηT, Rp, N_I, N_M; engine=zalm_engine)

    chahine_rates_numerical = chahine_rate_dist_entanglement_numerical.(μ, ηR, ηT, Rp, N_I, N_M; engine=chahine_engine)

    Plots.plot(μ, zalm_rates_numerical, label="ZALM Re (numerical)", xlabel="Mean Photon number μ", ylabel="Rate (bits/s)", legend=:bottomleft,color=1)
    
    Plots.plot!(μ, chahine_rates_numerical, label="CHAHINE Re (numerical)", xlabel="Number of input modes N_I", ylabel="Rate (bits/s)", color=3)
end
@time plot_rate_dist_entanglement_with_Nm()


MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...


So far this has been useful to model the base paper, Now with the use of Genqo one can model a diferent number of loss parameters.
specifically it is interesting to note that the probability of heralding for the chahine source decays slower than zalm. and zalm has higher probability of heralding only for mean photon number < 0.7 (TODO: CHECK ACTUAL NUMBER) so we can explore the results for a mean photon number that 

In [136]:
function plot_Pherald_comparison()
    engine_zalm = HybridProjectionEngine(8)
    engine_chahine = HybridProjectionEngine(6)
    μ = logrange(1e-4, 10, 100)

    ηR = 0.01
    ηT = 0.9

    PH_zalm_numerical = zalm_PHeralded_symbolic.(μ, ηR, ηT; engine=engine_zalm)
    PH_zalm_symbolic = zalm_PHeralded_symbolic.(μ, ηR, ηT)

    PH_chahine_numerical = chahine_PHeralded_numerical.(μ, ηR, ηT; engine=engine_chahine)
    PH_chahine_symbolic = chahine_PHeralded_symbolic.(μ, ηR, ηT)

    Plots.plot(μ, PH_zalm_symbolic, label = "ZALM analytic", xlabel = "Mean Photon Number",ylabel = "Heralding Probability", legend = :topright,color = 1)
    Plots.plot!(μ, PH_chahine_symbolic, label = "Chahine analytic", color = 3)
    Plots.plot!(μ, PH_zalm_numerical, label = "ZALM numerical (Genqo v2.0)", color = 2, linestyle = :dash)
    Plots.plot!(μ, PH_chahine_numerical, label = "Chahine numerical (Genqo v2.0)", color = 5, linestyle = :dash)
    Plots.title!("Heralding Probability Comparison for: \$\\eta_R = $(ηR),\\; \\eta_T = $(ηT)\$")
end
@time plot_Pherald_comparison()

MethodError: MethodError: no method matching zalm_PHeralded_symbolic(::Float64, ::Float64, ::Float64; engine::HybridProjectionEngine)
This method does not support all of the given keyword arguments (and may not support any).

Closest candidates are:
  zalm_PHeralded_symbolic(::Float64, ::Float64, ::Float64) got unsupported keyword argument "engine"
   @ Main ~/vscode/Genqo.jl/docs/tutorial/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y224sdnNjb2RlLXJlbW90ZQ==.jl:1


specifially it can be seen that \mu = 1/\etaT

In [137]:
using Plots

zalm_engine = HybridProjectionEngine(8)
chahine_engine = HybridProjectionEngine(6)
Rp = 10e9
N_I = 10
N_M = 1
ηR = 0.01
ηT_vals = 0.7:0.05:1.0
μ_vals = logrange(1e-4, 1.5, 100)

R_zalm = [zalm_rate_dist_entanglement_numerical.(μ, ηR, ηT, Rp, N_I, N_M; engine=zalm_engine) for μ in μ_vals, ηT in ηT_vals]
R_chahine = [chahine_rate_dist_entanglement_numerical.(μ, ηR, ηT, Rp, N_I, N_M; engine=chahine_engine) for μ in μ_vals, ηT in ηT_vals]

p1 = Plots.heatmap(ηT_vals, μ_vals, R_zalm;
    xlabel="η_T", ylabel="μ", title="RE zalm",
    yscale=:log10, color=:viridis, colorbar_title="F")

p2 = Plots.heatmap(ηT_vals, μ_vals, R_chahine;
    xlabel="η_T", ylabel="μ", title="chahine",
    yscale=:log10, color=:viridis, colorbar_title="B")

Plots.plot(p1, p2, layout=(1,2), size=(1000,600))

MethodError: MethodError: no method matching apply!(::GaussianState{QuadBlockBasis{Int64}, Vector{Float64}, Matrix{Float64}}, ::Vector{Int64}, ::GaussianUnitary{QuadBlockBasis{Int64}, BitVector, BitMatrix})
The function `apply!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  apply!(!Matched::Operator, ::Any, !Matched::T) where T<:QuantumInterface.AbstractSuperOperator
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:30
  apply!(!Matched::Operator, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:20
  apply!(!Matched::Ket, ::Any, !Matched::Operator)
   @ QuantumOpticsBase ~/.julia/packages/QuantumOpticsBase/3qPKu/src/apply.jl:14
  ...
